In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/student_habits_cleaned.csv")

df.head()

,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score
0,23,Female,0.0,1.2,1.1,No,85.0,8.0,Fair,6,Master,Average,8,Yes,56.2
1,20,Female,6.9,2.8,2.3,No,97.3,4.6,Good,6,High School,Average,8,No,100.0
2,21,Male,1.4,3.1,1.3,No,94.8,8.0,Poor,1,High School,Poor,1,No,34.3
3,23,Female,1.0,3.9,1.0,No,71.0,9.2,Poor,4,Master,Good,1,Yes,26.8
4,19,Female,5.0,4.4,0.5,No,90.9,4.9,Fair,3,Master,Good,1,No,66.4


In [2]:
# Create performance categories based on exam score
# This converts a regression problem into a classification problem

def performance_category(score):

    # Students scoring 80 or above
    if score >= 80:
        return "High Performer"

    # Students scoring between 60 and 79
    elif score >= 60:
        return "Average Performer"

    # Students scoring below 60
    else:
        return "At Risk"

# Create new target column
df["performance_category"] = df["exam_score"].apply(performance_category)

# Check results
df[["exam_score", "performance_category"]].head()

,exam_score,performance_category
0,56.2,At Risk
1,100.0,High Performer
2,34.3,At Risk
3,26.8,At Risk
4,66.4,Average Performer


In [3]:
# Count students in each performance category

df["performance_category"].value_counts()

performance_category
Average Performer    443
At Risk              280
High Performer       277
Name: count, dtype: int64

In [4]:
# Create risk levels for intervention analysis

def risk_level(score):

    # Students performing well
    if score >= 80:
        return "Low Risk"

    # Students performing moderately
    elif score >= 60:
        return "Medium Risk"

    # Students requiring academic support
    else:
        return "High Risk"

# Create risk level column
df["risk_level"] = df["exam_score"].apply(risk_level)

# Display results
df[["exam_score", "risk_level"]].head()

,exam_score,risk_level
0,56.2,High Risk
1,100.0,Low Risk
2,34.3,High Risk
3,26.8,High Risk
4,66.4,Medium Risk


In [5]:
# Student ID does not contribute to prediction
# Therefore it should be removed

if "student_id" in df.columns:
    df = df.drop("student_id", axis=1)

# Display updated dataset
df.head()

,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score,performance_category,risk_level
0,23,Female,0.0,1.2,1.1,No,85.0,8.0,Fair,6,Master,Average,8,Yes,56.2,At Risk,High Risk
1,20,Female,6.9,2.8,2.3,No,97.3,4.6,Good,6,High School,Average,8,No,100.0,High Performer,Low Risk
2,21,Male,1.4,3.1,1.3,No,94.8,8.0,Poor,1,High School,Poor,1,No,34.3,At Risk,High Risk
3,23,Female,1.0,3.9,1.0,No,71.0,9.2,Poor,4,Master,Good,1,Yes,26.8,At Risk,High Risk
4,19,Female,5.0,4.4,0.5,No,90.9,4.9,Fair,3,Master,Good,1,No,66.4,Average Performer,Medium Risk


In [6]:
# Features (input variables)

X = df.drop(
    ["exam_score", "performance_category", "risk_level"],
    axis=1
)

# Target for regression model
# Predict exact exam score

y_regression = df["exam_score"]

# Target for classification model
# Predict student category

y_classification = df["performance_category"]

In [7]:
# Identify categorical columns

categorical_cols = X.select_dtypes(
    include=["object"]
).columns

# Identify numerical columns

numerical_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns

print("Categorical Features:")
print(categorical_cols)

print("\nNumerical Features:")
print(numerical_cols)

Categorical Features:
Index(['gender', 'part_time_job', 'diet_quality', 'parental_education_level',
       'internet_quality', 'extracurricular_participation'],
      dtype='object')

Numerical Features:
Index(['age', 'study_hours_per_day', 'social_media_hours', 'netflix_hours',
       'attendance_percentage', 'sleep_hours', 'exercise_frequency',
       'mental_health_rating'],
      dtype='object')


In [8]:
# Machine Learning models cannot process text values
# Convert categorical variables into numerical format

X_encoded = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True
)

# Display encoded dataset
X_encoded.head()

,age,study_hours_per_day,social_media_hours,netflix_hours,attendance_percentage,sleep_hours,exercise_frequency,mental_health_rating,gender_Male,gender_Other,part_time_job_Yes,diet_quality_Good,diet_quality_Poor,parental_education_level_High School,parental_education_level_Master,internet_quality_Good,internet_quality_Poor,extracurricular_participation_Yes
0,23,0.0,1.2,1.1,85.0,8.0,6,8,False,False,False,False,False,False,True,False,False,True
1,20,6.9,2.8,2.3,97.3,4.6,6,8,False,False,False,True,False,True,False,False,False,False
2,21,1.4,3.1,1.3,94.8,8.0,1,1,True,False,False,False,True,True,False,False,True,False
3,23,1.0,3.9,1.0,71.0,9.2,4,1,False,False,False,False,True,False,True,True,False,True
4,19,5.0,4.4,0.5,90.9,4.9,3,1,False,False,False,False,False,False,True,True,False,False


In [9]:
# Check number of rows and columns
# Number of columns increases after encoding

X_encoded.shape

(1000, 18)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [10]:
# Import train_test_split for splitting data into training and testing sets
from sklearn.model_selection import train_test_split

# Split data for regression task
# Regression target: exact exam_score prediction
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_encoded,
    y_regression,
    test_size=0.20,
    random_state=42
)

# Split data for classification task
# Classification target: performance_category prediction
# stratify keeps class proportions similar in train and test sets
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_encoded,
    y_classification,
    test_size=0.20,
    random_state=42,
    stratify=y_classification
)

# Check shapes
print("Regression Training Features:", X_train_reg.shape)
print("Regression Testing Features:", X_test_reg.shape)
print("Classification Training Features:", X_train_cls.shape)
print("Classification Testing Features:", X_test_cls.shape)

Regression Training Features: (800, 18)
Regression Testing Features: (200, 18)
Classification Training Features: (800, 18)
Classification Testing Features: (200, 18)


In [11]:
from sklearn.preprocessing import StandardScaler

# Create scaler object
scaler = StandardScaler()

# Create copies of datasets

X_train_reg_scaled = X_train_reg.copy()
X_test_reg_scaled = X_test_reg.copy()

X_train_cls_scaled = X_train_cls.copy()
X_test_cls_scaled = X_test_cls.copy()

# Scale numerical features only

X_train_reg_scaled[numerical_cols] = scaler.fit_transform(
    X_train_reg[numerical_cols]
)

X_test_reg_scaled[numerical_cols] = scaler.transform(
    X_test_reg[numerical_cols]
)

# Scale classification data

X_train_cls_scaled[numerical_cols] = scaler.fit_transform(
    X_train_cls[numerical_cols]
)

X_test_cls_scaled[numerical_cols] = scaler.transform(
    X_test_cls[numerical_cols]
)

In [12]:
# Save encoded features

X_encoded.to_csv(
    "../data/processed/features_encoded.csv",
    index=False
)

# Save regression target

y_regression.to_csv(
    "../data/processed/target_regression.csv",
    index=False
)

# Save classification target

y_classification.to_csv(
    "../data/processed/target_classification.csv",
    index=False
)

# Save complete feature engineered dataset

df.to_csv(
    "../data/processed/student_habits_feature_engineered.csv",
    index=False
)

print("Processed datasets saved successfully!")

Processed datasets saved successfully!


In [13]:
# Check final dimensions

print("Feature Matrix Shape:", X_encoded.shape)

print("\nRegression Target Shape:")
print(y_regression.shape)

print("\nClassification Target Shape:")
print(y_classification.shape)

print("\nPerformance Category Distribution:")
print(y_classification.value_counts())

Feature Matrix Shape: (1000, 18)

Regression Target Shape:
(1000,)

Classification Target Shape:
(1000,)

Performance Category Distribution:
performance_category
Average Performer    443
At Risk              280
High Performer       277
Name: count, dtype: int64
